In [3]:
from pathlib import Path
import logfire
from pydantic_ai import Agent
from dotenv import load_dotenv

import requests
# from pydantic_ai_harness.subagents import SubAgent, SubAgents
# from pydantic_ai_harness.planning import Planning
# from pydantic_ai.common_tools.tavily import tavily_search_tool
load_dotenv()

# api_key = "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"

# PROMPTS_DIR = Path("prompts")


True

In [2]:
logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-eu.pydantic.dev/johanidler/rain

In [ ]:
import base64, uuid
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes, serialization

BASE_URL = "https://api-dev.raincards.xyz/v1"
headers = {"Api-Key": "c7f3672e9b88f1b69ef42f2a7ddf81f4a6a10cf1"}
teamId = "25caa5cd-ffe8-4974-a8a5-76e1113b9f5a"
userId = "7eeea853-ca13-4540-bf58-0e2c686a52dd"
contractId = "ee4df870-e3ec-4151-8144-7985eab60ccf"

# Rain sandbox public key, used only to encrypt the sessionid header below
SANDBOX_PUBLIC_KEY = b"""-----BEGIN PUBLIC KEY-----
MIGfMA0GCSqGSIb3DQEBAQUAA4GNADCBiQKBgQCAP192809jZyaw62g/eTzJ3P9H
+RmT88sXUYjQ0K8Bx+rJ83f22+9isKx+lo5UuV8tvOlKwvdDS/pVbzpG7D7NO45c
0zkLOXwDHZkou8fuj8xhDO5Tq3GzcrabNLRLVz3dkx0znfzGOhnY4lkOMIdKxlQb
LuVM/dGDC9UpulF+UwIDAQAB
-----END PUBLIC KEY-----"""

# 1. Fund collateral
x = requests.post(f"{BASE_URL}/simulate/collateral/fund", headers=headers, json={"contractId": contractId, "currency": "rusd", "amount": 100000})

# 2. Issue a scoped card (sessionid must be RSA-OAEP encrypted, not a plain string)
secret = base64.b64encode(bytes.fromhex(uuid.uuid4().hex))
pubkey = serialization.load_pem_public_key(SANDBOX_PUBLIC_KEY)
sessionid = base64.b64encode(pubkey.encrypt(secret, padding.OAEP(mgf=padding.MGF1(hashes.SHA1()), algorithm=hashes.SHA1(), label=None))).decode()

y = requests.post(f"{BASE_URL}/issuing/users/{userId}/cards/scoped", headers={**headers, "sessionid": sessionid}, json={"amountInUSDCents": 4299})
cardId = y.json()["id"]

# 3. Authorize a transaction
z = requests.post(f"{BASE_URL}/simulate/transactions/authorize", headers=headers, json={"cardId": cardId, "amount": 4299, "currency": "usd", "merchantName": "Demo Store", "merchantCategoryCode": "5999"})

# 3b. Settle the transaction
a = requests.post(f"{BASE_URL}/simulate/transactions/{z.json()['transactionId']}/settle", headers=headers)

# 4. Read transactions back
b = requests.get(f"{BASE_URL}/issuing/transactions?limit=20", headers=headers)

# 5. Create a payment route
c = requests.post(f"{BASE_URL}/payment-routes", headers=headers, json={"userId": userId, "source": {"currency": "usd", "rail": "ach"}, "destination": {"currency": "usdc", "rail": "base", "address": {"type": "onchain", "address": "0x742d35Cc6634C0532925a3b844Bc029e4e6C8bBd"}}})

# 5b. Simulate the payment route (max $100 per simulated transfer)
d = requests.post(f"{BASE_URL}/simulate/payment-routes", headers=headers, json={"paymentRouteId": c.json()["id"], "amount": 50})


In [4]:
travel_agent = Agent(
    model=model_gemini_retry,
    instructions=load_instructions("travel_agent"),
    capabilities=[
        Planning(),
        SubAgents(agents=[
            SubAgent(create_booking_agent),
            SubAgent(reserach_hotels_agent),
            SubAgent(research_flights_agent),
        ])
    ],

)